# Understanding ONNX | Part 1 — Create the model artifacts

**Train once, export for deployment.** Imagine a temperature conversion application. We know the formula `F = 1.8 × C + 32`, but use a tiny regression model to learn how a model moves from training to deployment. The original synthetic CSV has 100 examples. This is a teaching exercise, not a reason to replace a simple formula with ML.

## 1. Locate the dataset and output directory

In [ ]:
from pathlib import Path

# Find the repository whether Jupyter starts in the root or this part's folder.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data" / "temperature_conversion.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the tutorial repository.")
MODELS = ROOT / "models"
JOBLIB_PATH = MODELS / "linear_regression.joblib"
ONNX_PATH = MODELS / "linear_regression.onnx"

## 2. Learn the relationship
Train on 80% of the data and evaluate on the held-out 20%. MAE measures error in Fahrenheit; R² measures fit. A perfect linear relationship should yield nearly zero error.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

data = pd.read_csv(ROOT / "data" / "temperature_conversion.csv")
# One feature per row; use a two-dimensional array for the model input.
X = data[["input"]].to_numpy(dtype=np.float64)
y = data["output"].to_numpy(dtype=np.float64)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = LinearRegression().fit(X_train, y_train)
predictions = model.predict(X_test)
print(f"Coefficient: {model.coef_[0]:.6f}; intercept: {model.intercept_:.6f}")
print(f"Test MAE: {mean_absolute_error(y_test, predictions):.8f}")
print(f"Test R²: {r2_score(y_test, predictions):.6f}")
np.testing.assert_allclose(predictions, y_test, atol=1e-9, rtol=0)

## 3. Persist the Python model with Joblib
Joblib serializes the fitted Python estimator. Loading it requires a compatible scikit-learn environment. Never load an untrusted Joblib file.

In [ ]:
MODELS.mkdir(exist_ok=True)
joblib.dump(model, JOBLIB_PATH)
# Reload our own trusted artifact to verify that serialization preserves predictions.
loaded_model = joblib.load(JOBLIB_PATH)
np.testing.assert_allclose(loaded_model.predict(X_test), predictions, rtol=0, atol=0)
print(f"Saved {JOBLIB_PATH.name}: {JOBLIB_PATH.stat().st_size} bytes")

## 4. Convert the same fitted model to ONNX
ONNX is a standardized graph representation, not a runtime. `skl2onnx` translates the estimator into operators and learned parameters. The input contract is a float32 tensor of shape `[batch, 1]`. Structural validation does not prove numerical equivalence; Part 2 does that.

In [ ]:
import onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# None allows a variable batch size; 1 is the number of input features.
graph_model = convert_sklearn(
    loaded_model, initial_types=[("celsius", FloatTensorType([None, 1]))],
    target_opset=15,
)
# Validate the graph before publishing the artifact.
onnx.checker.check_model(graph_model)
onnx.save(graph_model, ONNX_PATH)
print(f"Saved {ONNX_PATH.name}: {ONNX_PATH.stat().st_size} bytes")

## 5. Look inside the graph
An operator describes computation; tensors carry values; an opset identifies operator semantics. The graph stores the learned coefficients, not the training dataset.

In [ ]:
graph_model = onnx.load(ONNX_PATH)
for opset in graph_model.opset_import:
    print("Opset:", opset.domain or "ai.onnx", opset.version)
for node in graph_model.graph.node:
    print("Operator:", node.domain or "ai.onnx", node.op_type)
for tensor in [*graph_model.graph.input, *graph_model.graph.output]:
    print(tensor.name, onnx.helper.printable_type(tensor.type))

## Next: deploy the artifacts
Both files are now available under `models/`. Part 2 opens them without training or conversion.